# Oppimisprojekti 3, osa 2: puheentunnistus Whisper-mallilla

Tässä notebookissa käytetään Hugging Facen `transformers`-kirjaston Whisper-malleja suomenkielisen puheen tunnistamiseen. Notebook lukee itse nauhoitetut alle 30 sekunnin äänitteet kansiosta `audio_samples` ja ajaa vertailun usealla mallikoolla.

**Tekoälyn käyttö:** Notebookin runko, analyysikysymykset ja raporttirakenne on laadittu Codex-avustajalla. Ääninäytteet on nauhoitettu itse, ja lopullinen arviointi täydennetään ajotulosten perusteella.

## Mel-spektrogrammi omin sanoin

Whisper ei syötä neuroverkolle raakaa ääniaaltoa. Ääni muunnetaan ensin mel-spektrogrammiksi.

- **x-akseli** kuvaa aikaa: vasemmalta oikealle edetään äänitteen alusta loppuun.
- **y-akseli** kuvaa taajuuksia mel-asteikolla. Mel-asteikko painottaa taajuuksia ihmiskuulon kannalta mielekkäämmin kuin lineaarinen hertsiasteikko.
- **väri** kuvaa voimakkuutta desibeleinä. Kirkkaammat tai voimakkaammat värit tarkoittavat, että kyseisellä ajanhetkellä ja taajuusalueella on enemmän energiaa.

Tämä esitysmuoto sopii neuroverkolle paremmin kuin raaka aalto, koska puheen olennaiset piirteet, kuten vokaalit, konsonantit ja äänenpainot, näkyvät paikallisina kuvioina ajan ja taajuuden tasossa. Transformer saa siis järjestetyn piirrejonon, jota se voi käsitellä samaan tapaan kuin osassa 1 tokenijonoa.

In [ ]:
# Tarvittaessa asenna riippuvuudet esimerkiksi:
# %pip install torch transformers librosa soundfile matplotlib pandas jiwer accelerate

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
from IPython.display import Audio, display
from transformers import WhisperProcessor, WhisperForConditionalGeneration

warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Laite:", device)

## Ääninäytteet

Notebook lukee äänitiedostot automaattisesti kansiosta `audio_samples`. Tässä projektissa käytössä on useita lyhyitä itse nauhoitettuja näytteitä: selkeä puhe, taustahäly, nopeampi tai epäselvempi puhe, murteellinen/puhekielinen puhe sekä englanninkielisiä vertailunäytteitä.

Kaikki nykyiset `.wav`-tiedostot ovat tarkistuksen perusteella 16 kHz mono -ääntä ja alle 30 sekuntia pitkiä, joten ne sopivat hyvin Whisperin kokeiluun.

Jos tiedät jonkin näytteen tarkan sanamuodon, lisää se `oikea_teksti`-kenttään. Silloin notebook voi laskea myös WER-virhemittarin.

In [ ]:
AUDIO_DIR = Path("audio_samples")
AUDIO_DIR.mkdir(exist_ok=True)

sample_descriptions = {
    "selkea_puhe.wav": {
        "kuvaus": "Selkeä suomenkielinen puhe, rauhallinen lukutapa.",
        "oikea_teksti": "",
    },
    "taustahaly.wav": {
        "kuvaus": "Suomenkielinen puhe taustahälyn kanssa.",
        "oikea_teksti": "",
    },
    "aaninayte_nopea.wav": {
        "kuvaus": "Nopeampi tai hieman epäselvempi suomenkielinen puhe.",
        "oikea_teksti": "",
    },
    "murre_aaninayte.wav": {
        "kuvaus": "Puhekielinen tai murteellinen suomenkielinen näyte.",
        "oikea_teksti": "",
    },
    "murre_aaninayte2.wav": {"kuvaus": "Toinen puhekielinen tai murteellinen suomenkielinen näyte.", "oikea_teksti": ""},
    "english_sample1.wav": {"kuvaus": "Englanninkielinen vertailunäyte.", "oikea_teksti": ""},
    "english_sample2.wav": {"kuvaus": "Englanninkielinen vertailunäyte.", "oikea_teksti": ""},
    "english_sample3.wav": {"kuvaus": "Englanninkielinen vertailunäyte.", "oikea_teksti": ""},
    "english_sample_fast.wav": {
        "kuvaus": "Nopea englanninkielinen vertailunäyte.",
        "oikea_teksti": "",
    },
    "selkea_puhe_1.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 1.", "oikea_teksti": ""},
    "selkea_puhe_2.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 2.", "oikea_teksti": ""},
    "selkea_puhe_3.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 3.", "oikea_teksti": ""},
    "selkea_puhe_4.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 4.", "oikea_teksti": ""},
    "selkea_puhe_5.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 5.", "oikea_teksti": ""},
}

supported = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}
audio_files = sorted([p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in supported])

print(f"Löytyi {len(audio_files)} äänitiedostoa kansiosta {AUDIO_DIR.resolve()}")
for p in audio_files:
    print("-", p.name)


In [ ]:
def load_audio(path, target_sr=16000):
    y, sr = librosa.load(path, sr=target_sr, mono=True)
    duration = librosa.get_duration(y=y, sr=sr)
    if duration > 30:
        print(f"Varoitus: {path.name} on {duration:.1f} s. Whisperin peruskokeeseen suositellaan alle 30 s pätkiä.")
    return y, sr, duration

for path in audio_files:
    y, sr, duration = load_audio(path)
    desc = sample_descriptions.get(path.name, {}).get("kuvaus", "Kuvaus puuttuu")
    print(f"{path.name}: {duration:.2f} s, {sr} Hz, {desc}")
    display(Audio(y, rate=sr))

## Mel-spektrogrammin visualisointi

Aja seuraava solu yhdelle tai useammalle näytteelle ja liitä havaintosi raporttiin. Kiinnitä huomiota siihen, missä kohtaa puhe, tauot ja mahdollinen taustahäly näkyvät.

In [ ]:
def plot_mel_spectrogram(path, n_mels=80):
    y, sr, duration = load_audio(path)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=160, n_fft=400)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title(f"Aaltomuoto: {path.name}")

    img = librosa.display.specshow(mel_db, sr=sr, hop_length=160, x_axis="time", y_axis="mel", ax=axes[1])
    axes[1].set_title("Mel-spektrogrammi")
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
    plt.tight_layout()
    plt.show()

if audio_files:
    plot_mel_spectrogram(audio_files[0])
else:
    print("Lisää äänitiedostoja audio_samples-kansioon ja aja solu uudelleen.")

## Whisper-mallien vertailu

Oletuksena vertaillaan malleja `tiny` ja `base`, koska ne latautuvat ja ajavat kohtuullisen nopeasti. Lisää listaan esimerkiksi `small` tai `large-v3`, jos käytössä on tarpeeksi muistia ja aikaa.

In [ ]:
MODEL_SIZES = ["tiny", "base"]  # voit kokeilla myos: "small", "medium", "large-v3"
LANGUAGE = "fi"
TASK = "transcribe"
MAX_SAMPLES = None  # vaihda arvoksi 1, jos haluat ensin nopean testiajon

def model_id(size):
    return "openai/whisper-large-v3" if size == "large-v3" else f"openai/whisper-{size}"


def load_whisper(size):
    name = model_id(size)
    print(f"Ladataan malli {name}...", flush=True)
    processor = WhisperProcessor.from_pretrained(name)
    model = WhisperForConditionalGeneration.from_pretrained(name).to(device)
    model.eval()
    print(f"Malli {name} ladattu.", flush=True)
    return processor, model


def transcribe(path, processor, model, language=LANGUAGE, task=TASK):
    y, sr, duration = load_audio(path)
    inputs = processor(y, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    start = time.perf_counter()
    with torch.no_grad():
        generated_ids = model.generate(input_features, language=language, task=task)
    seconds = time.perf_counter() - start

    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return text, duration, seconds


In [ ]:
results = []
files_to_run = audio_files[:MAX_SAMPLES] if MAX_SAMPLES else audio_files

print(f"Ajettavia tiedostoja: {len(files_to_run)}", flush=True)
print(f"Mallit: {', '.join(MODEL_SIZES)}", flush=True)

for size in MODEL_SIZES:
    print()
    print(f"Aloitetaan Whisper {size}...", flush=True)
    processor, whisper_model = load_whisper(size)

    for index, path in enumerate(files_to_run, start=1):
        print(f"  [{index}/{len(files_to_run)}] Tunnistetaan: {path.name}", flush=True)
        language = "en" if path.name.startswith("english") else "fi"
        prediction, duration, seconds = transcribe(path, processor, whisper_model, language=language)
        ref = sample_descriptions.get(path.name, {}).get("oikea_teksti", "")
        results.append({
            "malli": size,
            "tiedosto": path.name,
            "kieli": language,
            "kesto_s": round(duration, 2),
            "aika_s": round(seconds, 2),
            "reaaliaikakerroin": round(seconds / duration, 2) if duration else np.nan,
            "oikea_teksti": ref,
            "tunnistus": prediction,
        })
        print(f"    Valmis {seconds:.2f} sekunnissa", flush=True)

    del whisper_model
    if device == "cuda":
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df.to_csv("osa2_whisper_tulokset_oma_ajo.csv", index=False, encoding="utf-8-sig")
display(results_df)
print("Tulokset tallennettu tiedostoon osa2_whisper_tulokset_oma_ajo.csv", flush=True)


In [ ]:
# Valinnainen WER-laskenta.
# T?ss? ajossa oikeat sanasta sanaan -referenssit eiv?t ole tiedossa,
# joten WER j?tet??n pois. Jos kirjoitat tarkat puhutut lauseet
# sample_descriptions-sanastoon, voit ajaa t?m?n solun.

try:
    from jiwer import wer
    if not results_df.empty and results_df["oikea_teksti"].fillna("").str.len().gt(0).any():
        results_df["WER"] = results_df.apply(
            lambda r: wer(r["oikea_teksti"], r["tunnistus"]) if r["oikea_teksti"] else np.nan,
            axis=1,
        )
        display(results_df[["malli", "tiedosto", "kieli", "aika_s", "reaaliaikakerroin", "WER", "tunnistus"]])
    else:
        print("WER-laskentaa ei ajettu, koska tarkkoja referenssitekstej? ei ole t?ytetty.")
except Exception as exc:
    print("WER-laskentaa ei ajettu:", exc)

## Raportointitaulukko ja tulokset

Vertailu ajettiin kaikilla `audio_samples`-kansion 14 WAV-näytteellä ja kahdella Whisper-mallilla: `tiny` ja `base`. Ajo tehtiin CPU:lla. Aineistossa oli 10 suomenkielistä ja 4 englanninkielistä näytettä.

Tarkkoja sanasta sanaan kirjoitettuja referenssitekstejä ei ole näille äänitteille talletettu, joten arviointi tehdään tässä laadullisesti vertaamalla mallien tuottamia transkriptioita ja mittaamalla nopeutta. WER-lukua ei raportoida, koska väärä referenssiteksti antaisi harhaanjohtavan tuloksen.

| Ääninäyte | Kesto | tiny aika | base aika | tiny tunnistus | base tunnistus |
|---|---:|---:|---:|---|---|
| `aaninayte_nopea.wav` | 5.91 s | 1.15 s | 1.46 s | Puhulvahan nopeammin ja epäselvämin, jolla nähdään kuinka hyvin puheen tunnistustoimin kuin aantaminen ja... | Puhun vähän nopeammin ja epäselvämin, jotta nähdään kuinka hyvin puheen tunnistus toimii kuin aantaminen... |
| `english_sample1.wav` | 8.81 s | 0.61 s | 0.77 s | T.C. is a short English-peed sample for testing how well the program recognizes another language. | This is a short English speech sample for testing how well the program recognizes another language. |
| `english_sample2.wav` | 9.24 s | 0.44 s | 0.82 s | This is a short English speed sample for testing how well whisper recognizes another language. | This is a short English speech sample for testing how well Whisper recognizes another language. |
| `english_sample3.wav` | 6.08 s | 0.53 s | 0.90 s | This is a short English speech sample for testing how well these per record passes on a trot language. | is a short English speech sample for testing how well we use per recognition as a natural language. |
| `english_sample_fast.wav` | 4.71 s | 0.61 s | 0.91 s | This is a shorting speed sample for testing how well vis-pro record passes on other language. | This is a short English speech sample for testing how a Whisper recognizes another language. |
| `murre_aaninayte.wav` | 4.37 s | 0.67 s | 1.02 s | hieni histukseen, koita sanon sanot niin seläkestekku ossoa. | ja hissuksia ja koita sanot niin seläkästikko ossaan. |
| `murre_aaninayte2.wav` | 4.97 s | 0.72 s | 1.08 s | Mietpuhun ihan hisuksia ja koetaan sanossa on, että mahdollisimman seläkestin. | Mien puhun ihan hissukse ja koitan sanon sanat mahdollisimman seläkästi. |
| `selkea_puhe.wav` | 9.75 s | 1.24 s | 1.46 s | Tämä on selkeä Suomen kielenen äänen äänenäytä. Kiinit on huomioita siihen, että puhuntasaisesti ja ääntä... | Tämä on selkeä Suomen kielinen aanninäötä. Kiinnit on huomioita siihen, että puhun tasaisesti ja aantamin... |
| `selkea_puhe_1.wav` | 15.00 s | 1.52 s | 1.87 s | Suomen kielikuluurallilaisen kielikuntaan ja sen suomalaisukkilaisen haaran itäveren suomalaisiin kieliin... | Suomen kieli, suomen kieli kuuluu uralilaisen kieli kuntaan ja sen suomalaisuukkrilaisen haaran itäveren... |
| `selkea_puhe_2.wav` | 15.00 s | 0.55 s | 1.77 s | Suomen keli on tyydyksen saadaan luon mittaan suuri. | Kielisiä väestöryhmää on Pohjois-Ruotsissa, meän kielen puhujat, norjassa, kvenit, sekä inkerissä, missä... |
| `selkea_puhe_3.wav` | 15.00 s | 1.54 s | 1.83 s | Uudenpia Suomalaisuithmiä on syntynyt elroppaan sekä etenkin tuohat kadeksansa taluvulla pohjoisamerikkaa... | Tätä uudenpia Suomalaislyhmia on syntynyt Eurooppaan sekä etenkin 1800-luvulla Pohjoisamerikkaan, jossa s... |
| `selkea_puhe_4.wav` | 15.00 s | 1.39 s | 1.86 s | muuttoliikkentuloksena ruotsiin. Suomen kieli on voimakkaasti tai pova, algelutinoivakielin. Sanajärjesty... | muuttodikentulaksena ruotsii. Suomen kieli on voimakkaasti taipova, algolutinoiva kieli. Sana järjestys o... |
| `selkea_puhe_5.wav` | 15.00 s | 1.19 s | 1.92 s | Suomen kielikuluurailaisen kielikuntaan ja sen suomalaisuukkilaisen haram itäveren suomalaisiin kieliin S... | Suomen kieli, suomen kieli kuuluu uralilaisen kielikuntaan ja sen suomalaisuukkrilaisen haaran itäveren s... |
| `taustahaly.wav` | 7.70 s | 0.96 s | 1.44 s | Tämä on näytä on lautettu taustahely on kanssa. Lua on testotat tunnistakohjema puheen myös vaikea, mutta... | Tämä on näytöön nauhoitettu taustahälön kanssa. Alue on testata tunnistakouhelman puheen myös vaikeamassa... |

Keskiarvot:

| Malli | Keskimääräinen ajoaika | Keskimääräinen reaaliaikakerroin |
|---|---:|---:|
| tiny | 0.94 s | 0.11 |
| base | 1.36 s | 0.15 |

### Analyysi

Kun mallit oli ladattu välimuistiin, koko 14 näytteen ajo oli paikallisella CPU:lla nopea. `tiny` oli nopeampi: keskimäärin noin 0.94 sekuntia per näyte. `base` oli hitaampi, noin 1.36 sekuntia per näyte, mutta tuotti useissa kohdissa luettavampaa ja oikeamman näköistä tekstiä.

Suomenkielisissä näytteissä pienet Whisper-mallit tunnistivat usein aiheen ja lauserakenteen, mutta tekivät paljon kirjoitusvirheitä ja väärinkuulemisia. Pitkissä selkeissä lisänäytteissä `base` sai jo varsin käyttökelpoisia lauseita, esimerkiksi Suomen kieltä käsittelevissä näytteissä. Lyhyissä testilauseissa, taustahälyssä ja nopeassa puheessa virheitä jäi enemmän.

Englanninkieliset näytteet onnistuivat selvästi paremmin kuin suomenkieliset. Tämä näkyy erityisesti `english_sample2.wav`- ja `english_sample_fast.wav`-näytteissä, joissa `base` tuotti lähes virheettömän tekstin. Tämä on odotettavaa, koska Whisperin esikoulutusaineistossa englanti on erittäin vahvasti edustettuna.

Murteellinen tai puhekielinen puhe oli vaikeinta. Malli yritti normalisoida ääntä kirjakielen suuntaan tai tuotti sanoja, jotka kuulostavat suomen kaltaisilta mutta eivät vastaa puhetta täydellisesti. Taustahäly heikensi tunnistusta, mutta `base` säilytti yleensä enemmän lauseen rakenteesta kuin `tiny`.

### Yhteys osaan 1

Osassa 1 transformer käsitteli tekstin tokenijonoa ja ennusti seuraavaa tokenia. Whisperissä syöte ei ole tekstinä, vaan ääni muunnetaan ensin mel-spektrogrammiksi eli aika-taajuuspiirteiden jonoksi. Molemmissa tapauksissa transformer hyödyntää attention-mekanismia järjestetyn syötteen riippuvuuksien mallintamiseen. Tämä on transformerien vahvuus: sama perusidea sopii sekä tekstin että puheen kaltaisiin sekventiaalisiin aineistoihin.

## Vapaaehtoinen Gradio-käyttöliittymä

Jos haluat kokeilla mikrofonia selaimessa, poista kommentit ja aja solu. Tämä ei ole pakollinen palautukseen.

In [ ]:
# %pip install gradio
# import gradio as gr
#
# processor, whisper_model = load_whisper("base")
#
# def transcribe_gradio(audio_path):
#     if audio_path is None:
#         return ""
#     text, duration, seconds = transcribe(Path(audio_path), processor, whisper_model)
#     return f"{text}

#Kesto: {duration:.1f} s, tunnistusaika: {seconds:.1f} s"
#
# demo = gr.Interface(
#     fn=transcribe_gradio,
#     inputs=gr.Audio(sources=["microphone", "upload"], type="filepath"),
#     outputs=gr.Textbox(lines=6),
#     title="Whisper suomenkieliseen puheentunnistukseen",
# )
# demo.launch()